In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import gc
from tqdm import tqdm
from bioplnn.models import SpatiallyEmbeddedClassifier, SpatiallyEmbeddedRNN
from bioplnn.utils import (
    initialize_dataloader,
    initialize_scheduler,
    manual_seed,
)
import pickle
import matplotlib.pyplot as plt
checkpoint_path = "./train/checkpoints/"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

maze_data_path = "./data/mazes/"
checkpoint_path = "./train/checkpoints/"

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=4, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=5, padding=2)
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.gap   = nn.AdaptiveAvgPool2d(1)     # <- NEW
        self.fc1   = nn.Linear(128, 512)          # <- 64 channels only
        self.fc2   = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.gap(x)                          # [B, 64, 1, 1]
        x = torch.flatten(x, 1)                  # [B, 64]
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)

def prepare_rnn_weights(state_dict):
    """Remove 'rnn.' prefix from all keys in the state dict.
        Also remove any key starting with readout"""
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('rnn.'):
            new_key = key[4:]  # Remove 'rnn.' prefix
            new_state_dict[new_key] = value
        elif not key.startswith('readout'):
            new_state_dict[key] = value
    return new_state_dict
    
    
def load_model_and_config(wandb_name, checkpoint_path):
    """Load model configuration and instantiate models."""
    try:
        full_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        model_cfg, num_steps = full_cfg["model_config"], full_cfg["num_steps"]
    except:
        model_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        num_steps = 20
        
    try:
        if full_cfg["model_type"] == "cnn":
            model = SimpleCNN(in_channels=model_cfg["in_channels"], num_classes=model_cfg["num_classes"])
            classifier = SimpleCNN(in_channels=model_cfg["in_channels"], num_classes=model_cfg["num_classes"])
        else:
            model = SpatiallyEmbeddedRNN(**model_cfg["rnn_kwargs"])
            classifier = SpatiallyEmbeddedClassifier(**model_cfg)
    except:
        model = SpatiallyEmbeddedRNN(**model_cfg["rnn_kwargs"])
        classifier = SpatiallyEmbeddedClassifier(**model_cfg)

        # ——— Load checkpoint & weights ———
    try:
        state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
    except:
        state_dict = torch.load(checkpoint_path + f"{wandb_name}/{checkpoint}.pth", map_location=torch.device('cpu'))

    try:
        state_dict = state_dict["model_state"]
    except:
        pass

    model.load_state_dict(prepare_rnn_weights(state_dict))
    classifier.load_state_dict(state_dict)

    return model, classifier, num_steps, full_cfg

In [ ]:
def random_two_one_squares(x: torch.Tensor, *, tol: float = 0.0, gen=None):
    """
    Find two non-overlapping 2x2 squares of 1s in x and return:
      - dot_input: a zero-like mask with the two 2x2 squares set to 1
      - squares: list of length 2, each an (4, 2) tensor of (row, col) indices
    """
    if x.dim() != 2:
        raise ValueError("x must be a 2D tensor (H, W).")

    # cells equal to 1 (optionally within tolerance for floats)
    mask = (x == 1) if tol == 0 else ((x >= 1 - tol) & (x <= 1 + tol))

    H, W = mask.shape
    if H < 2 or W < 2:
        raise ValueError("x must be at least 2x2 to contain a 2x2 square.")

    # True where the 2x2 window (top-left at (i,j)) is all ones
    tl = mask[:-1, :-1]
    tr = mask[:-1,  1:]
    bl = mask[ 1:, :-1]
    br = mask[ 1:,  1:]
    top_left_ok = tl & tr & bl & br  # shape (H-1, W-1)

    candidates = torch.nonzero(top_left_ok, as_tuple=False)  # [(i,j) top-lefts], shape [K, 2]
    K = candidates.size(0)
    if K < 2:
        raise ValueError("Need at least two 2x2 squares of ones (non-overlapping).")

    # Randomize candidate order
    if gen is None:
        perm = torch.randperm(K, device=candidates.device)
    else:
        perm = torch.randperm(K, generator=gen, device=candidates.device)
    candidates = candidates[perm]

    # Helper to test overlap between two 2x2 squares given their top-lefts (i1,j1) and (i2,j2)
    # They overlap iff |i1 - i2| < 2 AND |j1 - j2| < 2 (they share at least one cell).
    def non_overlapping(a, b):
        return not (abs(int(a[0]) - int(b[0])) < 2 and abs(int(a[1]) - int(b[1])) < 2)

    # Pick first, then find a non-overlapping second
    first = candidates[0]
    second = None
    for c in candidates[1:]:
        if non_overlapping(first, c):
            second = c
            break

    if second is None:
        raise ValueError("Found multiple 2x2 squares, but none are non-overlapping. Try relaxing tol or input.")

    # Build output mask and explicit indices
    dot_input = torch.zeros_like(x, dtype=x.dtype)
    squares = []
    for tlrc in (first, second):
        i, j = int(tlrc[0]), int(tlrc[1])
        coords = torch.tensor([[i, j],
                               [i, j+1],
                               [i+1, j],
                               [i+1, j+1]], device=x.device, dtype=torch.long)
        squares.append(coords)
        dot_input[i, j] = 1
        dot_input[i, j+1] = 1
        dot_input[i+1, j] = 1
        dot_input[i+1, j+1] = 1

    return dot_input, squares

In [ ]:
## CORRELATED DOTS ##

# wandb_name, checkpoint = "fancy-silence-24", 370
# wandb_name, checkpoint = "glowing-deluge-31", 330

## MAZES ##

wandb_name, checkpoint = "golden-universe-515", 320 # ReLU, with I->I recurrence

# ——— Load model config & instantiate ———
model, classifier, num_steps, cfg = load_model_and_config(wandb_name, checkpoint_path)

model.eval()
model.to(device)

classifier.eval()
classifier.to(device)

def activation_helper(model, num_samples, dataloader, type):
    num_save_steps = 20

    # num_samples = len(dataloader.dataset) if hasattr(dataloader, 'dataset') else sum(1 for _ in dataloader)
    activations = np.zeros((num_samples, num_save_steps, (4 if type=="i" else 8), 48, 48), dtype=np.float32)

    with torch.no_grad():
        # # record activations
        sample_idx = 0
        for batch_inputs, label in tqdm(dataloader):
            # input: (batch_size, channels, H, W)
            batch_size = batch_inputs.shape[0]
            batch_inputs = batch_inputs.to(device)
            batch_output_states, batch_neuron_states, batch_feedback_states = model(batch_inputs, num_steps=num_save_steps)

            if type=="o":
                activity = batch_output_states[0].detach().cpu().numpy()
            else:
                activity = model.query_neuron_states(batch_neuron_states, 0, 0 if type=="e" else 1).detach().cpu().numpy()

            activations[sample_idx:sample_idx+batch_size, :, :, :, :] = activity
            sample_idx += batch_size
    
    return activations
    
def decision_helper(classifier, dataloader, decision_step):
    # # # record decisions
    decisions = []
    classifier.to(device)
    with torch.no_grad():
        for batch_inputs, label in tqdm(dataloader):
            # input: (batch_size, channels, H, W)
            batch_size = batch_inputs.shape[0]
            batch_inputs = batch_inputs.to(device)
            pred = classifier(batch_inputs, num_steps=decision_step)
            decisions.extend(torch.argmax(pred, dim=-1).detach().tolist())
    decisions = np.array(decisions)
    return decisions

def get_activations(model, classifier, num_samples, loader, decision_step):
    all_indices = np.arange(len(loader.dataset))
    subset_indices = np.random.choice(all_indices, size=num_samples, replace=False)

    # wrap dataset
    subset_dataset = Subset(loader.dataset, subset_indices)

    # create a new DataLoader
    dataloader = DataLoader(
        subset_dataset,
        batch_size=loader.batch_size,
        shuffle=False,              # usually keep consistent
        num_workers=loader.num_workers,
        pin_memory=True if hasattr(loader, "pin_memory") else False
    )

    e_a = activation_helper(model, num_samples, dataloader, "e")
    i_a = activation_helper(model, num_samples, dataloader, "i")
    o_a = activation_helper(model, num_samples, dataloader, "o")
    decisions = decision_helper(classifier, dataloader, decision_step)
    return e_a, i_a, o_a, decisions

In [ ]:
def plot_time_to_solution():

    batch_size = 1
    train_loader, test_loader = initialize_dataloader(
        seed=42, root=maze_data_path, batch_size=batch_size, dataset="mazes"
    )
    all_samples = list(train_loader)
    num_samples = 1000

    input = all_samples[3][0]
    inputs = []
    np.random.seed(0)
    for _ in range(num_samples):
        new_input = input[0].clone()
        new_input[-1] = random_two_one_squares(input[0,0])[0]
        inputs.append(new_input)
    inputs = torch.stack(inputs)

    close = inputs[19]
    mid = inputs[47]
    far = inputs[12]
    negative = inputs[49]

    classifier.to(device)
    logits_list = []
    for num_steps in range(1, 30):
        logits = classifier(mid.unsqueeze(0).to(device), num_steps=num_steps)
        logits_list.append(logits[0][1].detach().cpu().item()-logits[0][0].detach().cpu().item())
    
    plt.plot(logits_list)

In [ ]:
def plot_PCs(activations, decisions):
    

In [ ]:
batch_size = 512
num_samples = batch_size * 20
dataset = "mazes"
train_loader, test_loader = initialize_dataloader(
    seed=42, root=maze_data_path, batch_size=batch_size, dataset=dataset
)

decision_step = 20
train_ea, train_ia, train_oa, train_decisions = get_activations(model, classifier, num_samples, train_loader, decision_step)